# 01 · Reconstrucción de la serie S4 (Fix0 → Fix4)
Misma lógica del notebook `PRONOSTICO_S4_documentado_05102026_1212`, ahora importada desde `s4.reconstruction`. Salida: `data/processed/df_max_s4_reconstruido.csv` con las columnas de bandera `Fix0…Fix_Pico`, que la evaluación usa para **excluir los minutos reconstruidos** de las métricas.

In [1]:
import sys, warnings
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from s4.config import cargar_config, variante, ruta
from s4.reconstruction import *
cfg = cargar_config('../config/ablacion_2025_2026.yaml')

In [3]:
RUTA_CRUDO = ruta(cfg, "base") / "data/raw/df_max_s4_all_jic.csv"   # <-- ajustar
df = CargadorDatosS4(str(RUTA_CRUDO), montar_drive=False).cargar()
resumen_crudo = cobertura_diaria(df)
print(resumen_dias_sobre_umbral(resumen_crudo, 90))

{'dias_sobre_umbral': 1280, 'total_dias': 1397, 'porcentaje_sobre_umbral': np.float64(91.62491052254832)}


In [4]:
auditor0 = AuditorEventosS4(ConfigAuditoria())
episodios0, _ = auditor0.auditar(df)
df_fix0, bitacora0 = corregir_picos_aislados(df, episodios0, modo="media_local", flag_col="Fix0")
res = ejecutar_pipeline_s4(df_fix0, max_gap=180)
df_final = res["df4"]

[Fix0] Picos aislados corregidos: 8 episodios, 8 minutos (modo = 'media_local')
[Fix0] S4 original en esos minutos : [0.640, 1.209]
[Fix0] S4 asignado                 : [0.108, 0.153]
[Fix1] Días con cobertura en [98%, 100%): 846

RESULTADO (Fix1)
Filas de entrada              : 1,893,824
Filas incorporadas            : 12,122
  por método                  : {'ar': 12025, 'lineal': 97}
  rango S4 reconstruido       : [0.048, 0.415]
Huecos > 180 min sin reconstruir : 0 (0 minutos)
Ajustes AR descartados (no estacionarios): 0
Datos originales conservados  : True
Filas antes del filtro : 1,905,946
Filas después del filtro: 1,888,345
Días conservados       : 1,325
Días eliminados (<50%): 72
[Fix3] Días con cobertura en [90%, 100%): 32

RESULTADO (Fix3)
Filas de entrada              : 1,888,345
Filas incorporadas            : 1,980
  por método                  : {'ar': 1951, 'lineal': 29}
  rango S4 reconstruido       : [0.055, 0.452]
Huecos > 180 min sin reconstruir : 0 (0 minutos)
Ajuste

In [5]:
# Auditoría final y corrección de picos que pudieran quedar (Fix_Pico)
auditor = AuditorEventosS4(ConfigAuditoria())
episodios, _ = auditor.auditar(df_final)
df_final_auditado, bitacora = corregir_picos_aislados(df_final, episodios, modo="media_local", flag_col="Fix_Pico")
# OJO: el notebook anterior guardaba df_final (sin Fix_Pico). Aquí se guarda la versión auditada.
salida = ruta(cfg, "s4_reconstruido"); salida.parent.mkdir(parents=True, exist_ok=True)
df_final_auditado.to_csv(salida)
print("Guardado:", salida, f"{len(df_final_auditado):,} filas")

[Fix_Pico] Picos aislados corregidos: 0 episodios, 0 minutos (modo = 'media_local')
Guardado: /home/soporte/Documents/scintia-s4/data/processed/df_max_s4_reconstruido.csv 1,891,672 filas
